In [1]:
!pip install playwright beautifulsoup4 nest_asyncio -q
!playwright install chromium
!playwright install-deps chromium

In [2]:
import asyncio
import nest_asyncio
nest_asyncio.apply()
import random
import re
import subprocess
import platform

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright, TimeoutError as PWTimeout
from urllib.parse import urlparse, parse_qs, unquote

import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
SERVICES = {
    "Mobile App Developers":  "https://clutch.co/directory/mobile-application-developers",
    "Web Developers":          "https://clutch.co/directory/web-developers",
    "Software Developers":     "https://clutch.co/developers",
    "Ecommerce Developers":    "https://clutch.co/developers/ecommerce",
    "UI/UX Agencies":          "https://clutch.co/agencies/ui-ux",
    "SEO Firms":               "https://clutch.co/seo-firms",
    "IT Services":             "https://clutch.co/it-services",
}


In [4]:
def get_company_website(card):
    tag = card.select_one("a.provider__cta-link.website-link__item.website-link__item--non-ppc")
    if not tag:
        return ""
    href = tag.get("href", "")
    if not href:
        return ""
    qs = parse_qs(urlparse(href).query)
    if "u" in qs:
        return unquote(qs["u"][0])
    return href

def get_total_count(soup):
    tag = soup.select_one("p.navbar__companies-amount.directory-only-related-block")
    return tag.get_text(strip=True) if tag else "N/A"

def parse_html(html, service_name, category_url):
    if not html:
        return []
    soup = BeautifulSoup(html, "html.parser")
    print("  Total on Clutch: " + get_total_count(soup))
    provider_list = soup.select_one("ul.providers__list")
    if not provider_list:
        print("  No provider list found")
        return []
    cards = provider_list.select("li.provider-list-item")
    print("  Cards on this page: " + str(len(cards)))
    results = []
    for card in cards:
        try:
            name_tag = card.select_one("h3.provider__title")
            company_name = name_tag.get_text(strip=True) if name_tag else ""
            profile_tag = card.select_one('a[href^="/profile/"]')
            company_url = ("https://clutch.co" + profile_tag.get("href", "") if profile_tag else "")
            company_website = get_company_website(card)
            if company_name:
                results.append({
                    "company_name":    company_name,
                    "company_url":     company_url,
                    "company_website": company_website,
                    "source_category": service_name,
                })
        except Exception as e:
            print("  Skipped card: " + str(e))
    return results



In [5]:

import random

BROWSER_ARGS = [
    "--disable-blink-features=AutomationControlled",
    "--disable-infobars",
    "--window-size=1440,900",
]

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]

STEALTH_JS = """
Object.defineProperty(navigator,'webdriver',{get:()=>undefined});
Object.defineProperty(navigator,'plugins',{get:()=>[1,2,3,4,5]});
Object.defineProperty(navigator,'languages',{get:()=>['en-US','en']});
window.chrome={runtime:{}};
"""

def build_page_url(base_url, page_num):
    if page_num == 1:
        return base_url
    sep = "&" if "?" in base_url else "?"
    return base_url + sep + "page=" + str(page_num)

async def render_page(pw, url):
    print("  Opening : " + url)
    try:
        browser = await pw.chromium.launch(headless=True, args=BROWSER_ARGS)
        context = await browser.new_context(
            user_agent=random.choice(USER_AGENTS),
            viewport={"width": 1440, "height": 900},
            locale="en-US",
            timezone_id="America/New_York",
        )
        await context.add_init_script(STEALTH_JS)
        page = await context.new_page()

        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        for selector in ["ul.providers__list", "li.provider-list-item", "[class*='providers']"]:
            try:
                await page.wait_for_selector(selector, timeout=15000)
                break
            except PWTimeout:
                continue
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        await asyncio.sleep(1.5)
        html = await page.content()

        await context.close()
        await browser.close()
        return html
    except PWTimeout:
        print("  Timeout : " + url)
        return ""
    except Exception as e:
        print("  Error : " + str(e))
        return ""

async def run_extraction(selected_services, max_pages):
    all_companies = []
    seen_urls = set()
    total_services = len(selected_services)

    async with async_playwright() as pw:
        for i, (service_name, base_url) in enumerate(selected_services.items(), 1):
            print("\n[" + str(i) + "/" + str(total_services) + "] " + service_name)

            for p in range(1, max_pages + 1):
                url = build_page_url(base_url, p)
                print("\n  Page " + str(p) + "/" + str(max_pages))
                html = await render_page(pw, url)
                companies = parse_html(html, service_name, base_url)
                if not companies:
                    print("  No companies found - stopping pagination")
                    break
                for c in companies:
                    key = c.get("company_url") or c.get("company_name", "")
                    if key and key not in seen_urls:
                        seen_urls.add(key)
                        all_companies.append(c)
                print("  Running total: " + str(len(all_companies)))
                delay = random.uniform(2, 4)
                await asyncio.sleep(delay)

            if i < total_services:
                delay = random.uniform(3, 5)
                print("\n  Waiting " + str(round(delay, 1)) + "s before next category...")
                await asyncio.sleep(delay)

    return all_companies



In [6]:
extracted_companies = []
HEADER_HTML = """
<div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    border-radius: 12px;
    padding: 24px 28px;
    margin-bottom: 16px;
    border: 1px solid #0f3460;
">
    <h2 style="color:#e94560; margin:0 0 4px 0; font-size:22px; letter-spacing:1px;">
        🔍 CLUTCH SCRAPER
    </h2>
    <p style="color:#a8a8b3; margin:0; font-size:13px;">
        Select service categories and pages, then extract company listings from Clutch.co
    </p>
</div>
"""

SECTION_STYLE = """
<div style="
    background:#fafafa;
    border:1px solid #e0e0e0;
    border-radius:8px;
    padding:14px 18px 6px 18px;
    margin-bottom:12px;
">
    <p style="font-weight:600; font-size:14px; color:#333; margin:0 0 8px 0;">
        {title}
    </p>
</div>
"""
chk_style = {"description_width": "0px"}
chk_layout = widgets.Layout(width="220px", margin="2px 0")

service_checks = {
    name: widgets.Checkbox(
        value=False, description=name,
        style=chk_style, layout=chk_layout,
        indent=False,
    )
    for name in SERVICES
}

chk_list = list(service_checks.values())
col1 = widgets.VBox(chk_list[:4], layout=widgets.Layout(margin="0 32px 0 0"))
col2 = widgets.VBox(chk_list[4:])
chk_grid = widgets.HBox([col1, col2], layout=widgets.Layout(margin="0 0 0 8px"))

select_all = widgets.Checkbox(
    value=False, description="Select all",
    style=chk_style, layout=widgets.Layout(width="140px"),
    indent=False,
)

def toggle_all(change):
    for chk in service_checks.values():
        chk.value = change["new"]

select_all.observe(toggle_all, names="value")

pages_input = widgets.BoundedIntText(
    value=1, min=1, max=50,
    layout=widgets.Layout(width="70px"),
)
pages_row = widgets.HBox(
    [
        widgets.HTML('<span style="font-size:13px;color:#555;">Pages per service&nbsp;&nbsp;</span>'),
        pages_input,
        widgets.HTML('<span style="font-size:12px;color:#999;">&nbsp;(1 page = 50 companies)</span>'),
    ],
    layout=widgets.Layout(align_items="center", margin="0 0 0 8px"),
)
run_button = widgets.Button(
    description="▶  Extract Companies",
    button_style="success",
    layout=widgets.Layout(width="200px", height="38px"),
    style={"font_weight": "bold"},
)
clear_button = widgets.Button(
    description="✕  Clear Results",
    button_style="warning",
    layout=widgets.Layout(width="160px", height="38px"),
)
btn_row = widgets.HBox(
    [run_button, clear_button],
    layout=widgets.Layout(margin="4px 0 0 0", gap="10px"),
)
output_area = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        border_radius="8px",
        padding="12px",
        margin="12px 0 0 0",
        min_height="60px",
        max_height="500px",
        overflow_y="auto",
    )
)
def on_clear_clicked(b):
    global extracted_companies
    extracted_companies = []
    with output_area:
        clear_output()
        print("✓ Results cleared. Ready for new extraction.")


def on_run_clicked(b):
    global extracted_companies
    with output_area:
        clear_output()

        selected = {n: SERVICES[n] for n, c in service_checks.items() if c.value}
        max_pages = pages_input.value

        if not selected:
            print("⚠  Please select at least one service.")
            return

        total_est = max_pages * 50 * len(selected)
        print("=" * 55)
        print("  CLUTCH LEAD EXTRACTION")
        print("=" * 55)
        print(f"  Services : {', '.join(selected)}")
        print(f"  Pages    : {max_pages} per service (50 companies/page)")
        print(f"  Max      : ~{total_est} companies")
        print("=" * 55)

        loop = asyncio.get_event_loop()
        companies = loop.run_until_complete(
            run_extraction(selected, max_pages)
        )
        extracted_companies = companies

        print(f"\n{'=' * 55}")
        print(f"  ✓ DONE — Total extracted: {len(companies)}")
        print(f"{'=' * 55}")
        print("\n  ALL EXTRACTED COMPANIES:")
        print(f"  {'-' * 49}")
        for i, c in enumerate(companies, 1):
            print(f"  {i}. {c.get('company_name', '')}")
            print(f"     url      : {c.get('company_url', '')}")
            print(f"     website  : {c.get('company_website', '')}")
            print(f"     category : {c.get('source_category', '')}")
            print()


run_button.on_click(on_run_clicked)
clear_button.on_click(on_clear_clicked)

display(widgets.HTML(HEADER_HTML))
display(widgets.HTML('<p style="font-weight:600;font-size:14px;color:#333;margin:0 0 6px 0;">Service categories</p>'))
display(select_all)
display(chk_grid)
display(widgets.HTML('<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0;">'))
display(pages_row)
display(widgets.HTML('<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0;">'))
display(btn_row)
display(output_area)

HTML(value='\n<div style="\n    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);\n    border-rad…

HTML(value='<p style="font-weight:600;font-size:14px;color:#333;margin:0 0 6px 0;">Service categories</p>')

Checkbox(value=False, description='Select all', indent=False, layout=Layout(width='140px'), style=DescriptionS…

HTML(value='<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0;">')

HTML(value='<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0;">')

Output(layout=Layout(border='1px solid #ddd', margin='12px 0 0 0', max_height='500px', min_height='60px', over…

In [16]:
CONCURRENCY = 10
PAGE_TIMEOUT = 30000
SELECTOR_TIMEOUT = 8000
SCROLL_WAIT = 0.3
BLOCKED_RESOURCES = {"image", "media", "font", "stylesheet", "manifest", "other"}

BROWSER_ARGS = [
    "--disable-blink-features=AutomationControlled",
    "--disable-infobars",
    "--window-size=1440,900",
]
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
]
STEALTH_JS = """
Object.defineProperty(navigator,'webdriver',{get:()=>undefined});
Object.defineProperty(navigator,'plugins',{get:()=>[1,2,3,4,5]});
Object.defineProperty(navigator,'languages',{get:()=>['en-US','en']});
window.chrome={runtime:{}};
"""


def _parse_chart_data(cj):
    if not cj or not isinstance(cj, dict): return {}
    r = {}
    def ex(s):
        items = []
        for x in s.get("slices", []):
            n = x.get("name", "")
            p = x.get("PercentHundreds", 0) or round(x.get("percent", 0)*100)
            if n: items.append({"name": n, "pct": f"{p}%" if p else ""})
        return items
    for k, s in cj.items():
        if not isinstance(s, dict): continue
        if "slices" in s:
            r[k] = {"title": s.get("legend_title", k), "items": ex(s)}
        if "charts" in s and isinstance(s["charts"], dict):
            for sk, ss in s["charts"].items():
                if isinstance(ss, dict) and "slices" in ss:
                    r[f"{k}__{sk}"] = {"title": ss.get("legend_title", sk), "items": ex(ss)}
    return r

def _map_charts(ch):
    d = {"services": [], "industry": [], "client_type": [], "focus_area": []}
    key_map = {
        "services":    ["service_provided", "services"],
        "industry":    ["industries", "industry"],
        "client_type": ["clients", "client_focus", "client_type", "client", "client_size"],
    }
    for field, candidates in key_map.items():
        for ck in candidates:
            if ck in ch:
                d[field] = ch[ck]["items"]
                break
    for k, v in ch.items():
        if k.startswith("focus__"):
            tl = v["title"].lower()
            if any(kw in tl for kw in ["client focus", "client type", "client size"]):
                if not d["client_type"]:
                    d["client_type"] = v["items"]
            else:
                d["focus_area"].append({"category": v["title"], "items": v["items"]})
    title_map = {
        "services":    ["service line"],
        "industry":    ["industry"],
        "client_type": ["client focus", "client type", "client size"],
    }
    for field, keywords in title_map.items():
        if d[field]: continue
        for k, v in ch.items():
            tl = v["title"].lower()
            if any(kw in tl for kw in keywords):
                d[field] = v["items"]
                break
    return d

def _charts_from_legends(soup):
    d = {"services": [], "industry": [], "client_type": [], "focus_area": []}
    sections = soup.select("section.profile-chart--section, div.profile-chart__wrapper, div.profile-chart")
    for sec in sections:
        te = sec.select_one("div.chart-legend--title, h3, h4, [class*='chart-title']")
        t = te.get_text(strip=True).lower() if te else ""
        items = []
        for li in sec.select("li.chart-legend--item"):
            name_tag = li.select_one("a.chart-legend--item-link")
            name = name_tag.get_text(strip=True) if name_tag else li.get_text(strip=True)
            full = li.get_text(strip=True)
            pct_match = re.search(r'(\d+%)', full)
            pct = pct_match.group(1) if pct_match else ""
            if name: items.append({"name": name, "pct": pct})
        if not items: continue
        if any(x in t for x in ["service", "line"]): d["services"] = items
        elif "industry" in t: d["industry"] = items
        elif "client" in t: d["client_type"] = items
        else: d["focus_area"].append({"category": t.title(), "items": items})
    return d


def _find_desc(soup):
    for sel in ["div.profile-summary__text", "[class*='profile-summary__body']",
                "[class*='profile-about']", "[class*='field-name-body']", "[class*='company-about']"]:
        tag = soup.select_one(sel)
        if tag:
            paras = tag.find_all("p")
            text = " ".join(p.get_text(strip=True) for p in paras) if paras else tag.get_text(strip=True)
            if len(text) > 20: return text
    for h in soup.find_all(["h2", "h3", "h4"]):
        if "about" in h.get_text(strip=True).lower():
            s = h.find_next_sibling("p")
            if s and len(s.get_text(strip=True)) > 20: return s.get_text(strip=True)
    meta = soup.find("meta", attrs={"name": "description"})
    return meta["content"].strip() if meta and meta.get("content") else ""

def _rating(soup):
    for a in ["data-rating", "data-score", "aria-label"]:
        for t in soup.select("[class*='sg-rating']"):
            m = re.search(r'(\d+\.?\d*)', t.get(a, ""))
            if m: return m.group(1)
    for sel in ["[class*='rating__number']", "[class*='sg-rating__number']", "[class*='reviews-summary__rating']"]:
        t = soup.select_one(sel)
        if t:
            m = re.search(r'(\d+\.?\d*)', t.get_text(strip=True))
            if m: return m.group(1)
    for t in soup.select("[class*='scroll-to-review']"):
        m = re.search(r'(\d+\.?\d*)', t.get_text(strip=True))
        if m: return m.group(1)
    return ""

def _summary(soup):
    f = {"company_size": "", "project_min_cost": "", "hourly_rate": "", "year_founded": ""}
    for li in soup.select("li.profile-summary__detail"):
        le = li.select_one("span.profile-summary__detail-label")
        if not le: continue
        lt = le.get_text(strip=True).lower()
        ve = li.select_one("span.profile-summary__detail-title") or li.select_one("div.profile-summary__wrapper")
        if not ve: continue
        v = ve.get_text(strip=True)
        if not v: continue
        if "min" in lt and "project" in lt: f["project_min_cost"] = v
        elif "employee" in lt or ("size" in lt and "project" not in lt): f["company_size"] = v
        elif "hourly" in lt or "rate" in lt: f["hourly_rate"] = v
        elif "founded" in lt or "year" in lt:
            m = re.search(r'(\d{4})', v)
            f["year_founded"] = m.group(1) if m else v
    return f

def _locs(soup):
    r = []
    for h in soup.find_all("h2"):
        ht = h.get_text(strip=True)
        if re.match(r'^\d+\s+Location', ht):
            sib = h.find_next_sibling()
            if sib:
                children = sib.select("li, div, span, a")
                for c in children:
                    x = c.get_text(strip=True)
                    if x and "," in x and len(x) < 80 and x not in r:
                        r.append(x)
                if not r:
                    for line in sib.get_text("\n").split("\n"):
                        x = line.strip()
                        if x and "," in x and len(x) < 80 and x not in r:
                            r.append(x)
            if r:
                return r
    for li in soup.select("li.profile-summary__detail"):
        le = li.select_one("span.profile-summary__detail-label")
        if not le:
            continue
        if "location" in le.get_text(strip=True).lower():
            ve = (li.select_one("span.profile-summary__detail-title")
                  or li.select_one("div.profile-summary__wrapper"))
            if ve:
                x = ve.get_text(strip=True)
                if x and x not in r:
                    r.append(x)
    if r:
        return r
    for btn in soup.select("button.location-button"):
        x = btn.get_text(strip=True)
        if x and x.lower() != "headquarters" and x not in r:
            r.append(x)
    if r:
        return r
    for addr in soup.select("address.detailed-address"):
        x = addr.get_text(strip=True)
        if x and x not in r:
            r.append(x)
    return r


def parse_profile(html, cj, src):
    soup = BeautifulSoup(html, "html.parser")
    d = {
        "company_name": src.get("company_name", ""),
        "company_url": src.get("company_url", ""),
        "company_website": src.get("company_website", ""),
        "source_category": src.get("source_category", ""),
        "services": [], "industry": [], "client_type": [], "focus_area": [],
        "locations": [], "company_size": "", "description": "",
        "project_min_cost": "", "clutch_rating": "", "hourly_rate": "",
        "year_founded": "", "languages_services": [], "timezones_services": [],
    }
    n = soup.select_one("h1.profile-header__title")
    if n:
        s = n.find("small")
        if s: s.decompose()
        d["company_name"] = n.get_text(strip=True)
    d["clutch_rating"] = _rating(soup)
    d["description"] = _find_desc(soup)
    sm = _summary(soup)
    d["company_size"] = sm["company_size"]
    d["project_min_cost"] = sm["project_min_cost"]
    d["hourly_rate"] = sm["hourly_rate"]
    d["year_founded"] = sm["year_founded"]
    ch = _map_charts(_parse_chart_data(cj))
    if not ch["services"] and not ch["industry"]:
        ch = _charts_from_legends(soup)
    d["services"] = ch["services"]; d["industry"] = ch["industry"]
    d["client_type"] = ch["client_type"]; d["focus_area"] = ch["focus_area"]
    d["locations"] = _locs(soup)
    for h in soup.find_all(["h2","h3","h4","h5","dt","strong"]):
        if "language" in h.get_text(strip=True).lower():
            p = h.parent
            for _ in range(3):
                items = p.select("li")
                if items:
                    d["languages_services"] = [i.get_text(strip=True) for i in items if i.get_text(strip=True)]
                    break
                p = p.parent
                if p is None: break
            break
    for h in soup.find_all(["h2","h3","h4","h5","dt","strong"]):
        ht = h.get_text(strip=True).lower()
        if "timezone" in ht or "time zone" in ht:
            p = h.parent
            for _ in range(3):
                items = p.select("dd, li, span")
                if items:
                    d["timezones_services"] = [i.get_text(strip=True) for i in items if i.get_text(strip=True)]
                    break
                p = p.parent
                if p is None: break
            break
    return d


async def _block(route):
    if route.request.resource_type in BLOCKED_RESOURCES:
        await route.abort()
    else:
        await route.continue_()


async def fetch_one(browser, sem, idx, co, res, total, ctr):
    url = co.get("company_url", "")
    name = co.get("company_name", "Unknown")
    if not url:
        ctr[0] += 1
        print(f"  [{ctr[0]}/{total}] {name} — skipped")
        res[idx] = co
        return
    async with sem:
        ctx = None
        try:
            ctx = await browser.new_context(
                user_agent=random.choice(USER_AGENTS),
                viewport={"width": 1440, "height": 900},
                locale="en-US", timezone_id="America/New_York",
            )
            await ctx.add_init_script(STEALTH_JS)
            page = await ctx.new_page()
            await page.route("**/*", _block)
            await page.goto(url, wait_until="domcontentloaded", timeout=PAGE_TIMEOUT)
            try:
                await page.wait_for_selector("h1.profile-header__title", timeout=SELECTOR_TIMEOUT)
            except PWTimeout:
                pass
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(SCROLL_WAIT)
            html = await page.content()
            cj = None
            try: cj = await page.evaluate("window.chartPie || null")
            except: pass
            if html:
                pr = parse_profile(html, cj, co)
                res[idx] = pr
                filled = sum(1 for k, v in pr.items() if v and k not in ("company_url","company_website","source_category"))
                ctr[0] += 1
                print(f"  [{ctr[0]}/{total}] {name} — {filled}/14 fields")
            else:
                ctr[0] += 1
                print(f"  [{ctr[0]}/{total}] {name} — empty")
                res[idx] = co
        except PWTimeout:
            ctr[0] += 1
            print(f"  [{ctr[0]}/{total}] {name} — TIMEOUT")
            res[idx] = co
        except Exception as e:
            ctr[0] += 1
            print(f"  [{ctr[0]}/{total}] {name} — {type(e).__name__}")
            res[idx] = co
        finally:
            if ctx:
                try: await ctx.close()
                except: pass


async def run_stage2(companies):
    total = len(companies)
    print(f"  Concurrency {CONCURRENCY}, resource blocking ON, scroll 0.3s\n")

    # Kill stale chromium — skip on Windows
    if platform.system() != "Windows":
        try: subprocess.run(["pkill", "-9", "-f", "chromium"], timeout=5, capture_output=True)
        except: pass

    await asyncio.sleep(1)
    sem = asyncio.Semaphore(CONCURRENCY)
    res = {}
    ctr = [0]
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True, args=BROWSER_ARGS)
        tasks = [fetch_one(browser, sem, i, c, res, total, ctr) for i, c in enumerate(companies)]
        await asyncio.gather(*tasks)
        try: await browser.close()
        except: pass
    return [res[i] if i in res else companies[i] for i in range(total)]


print(f"Enriching {len(extracted_companies)} companies...\n")
loop = asyncio.get_event_loop()
enriched_companies = loop.run_until_complete(run_stage2(extracted_companies))
print(f"\nDONE — {len(enriched_companies)} profiles enriched")


# RETRY FAILED
failed = []
failed_idx = []
for i, c in enumerate(enriched_companies):
    filled = sum(1 for k, v in c.items() if v and k not in ("company_url","company_website","source_category"))
    if filled < 5:
        failed.append(c)
        failed_idx.append(i)

if failed:
    print(f"\nRetrying {len(failed)} failed companies...\n")
    retried = loop.run_until_complete(run_stage2(failed))
    for j, idx in enumerate(failed_idx):
        enriched_companies[idx] = retried[j]
    print(f"Retry done — patched {len(failed)} companies")
else:
    print("\nNo failed companies — all good!")



Enriching 50 companies...

  Concurrency 10, resource blocking ON, scroll 0.3s

  [1/50] IT-Geeks — 12/14 fields
  [2/50] Transform Agency — 14/14 fields
  [3/50] EUX Digital — 14/14 fields
  [4/50] Classy Llama — 14/14 fields
  [5/50] MageMontreal — 14/14 fields
  [6/50] Staylime — 14/14 fields
  [7/50] Codup — 14/14 fields
  [8/50] MOBIKASA — 14/14 fields
  [9/50] Ambaum — 14/14 fields
  [10/50] Elogic Commerce — 14/14 fields
  [11/50] DigitalSuits - Shopify Plus Partner — 14/14 fields
  [12/50] Superco — 13/14 fields
  [13/50] SPLIT Development - Shopify Plus Agency — 14/14 fields
  [14/50] Afocus — 14/14 fields
  [15/50] Commerce Pundit — 14/14 fields
  [16/50] Rave Digital — 14/14 fields
  [17/50] Polcode — 14/14 fields
  [18/50] Naturaily — 14/14 fields
  [19/50] FJ Solutions — 14/14 fields
  [20/50] ControlF5 — 14/14 fields
  [21/50] WPWeb Infotech — 14/14 fields
  [22/50] Digital Aptech — 14/14 fields
  [23/50] First Pier — 14/14 fields
  [24/50] Neuralab — 14/14 fields
  [25/5

In [17]:
import re

TECH_KEYWORDS = [
    "software development", "custom software", "software engineering",
    "application development", "app development",
    "web development", "web design", "web application",
    "mobile app development", "mobile development",
    "ios development", "android development", "cross-platform",
    "progressive web", "hybrid app",
    "full stack", "fullstack",
    "backend", "back-end", "frontend", "front-end",
    "mvp development", "prototype", "product development",
    "platform development", "api development",
    "microservices", "serverless",
    "saas", "paas",
    "flutter", "react native", "react", "angular", "vue",
    "node", "python", "java", "php", ".net", "ruby on rails",
    "swift", "kotlin", "typescript", "golang", "rust",
    "laravel", "django", "spring boot",
    "ecommerce development", "e-commerce development",
    "shopify development", "magento", "woocommerce",
    "bigcommerce", "opencart", "prestashop",
    "cms development", "wordpress development",
    "drupal", "joomla", "contentful", "strapi",
    "headless cms", "webflow development",
    "cloud", "cloud computing", "cloud migration", "cloud consulting",
    "aws", "azure", "gcp", "google cloud",
    "kubernetes", "docker", "terraform",
    "devops", "ci/cd", "infrastructure as code",
    "site reliability", "platform engineering",
    "ai ", "artificial intelligence", "generative ai",
    "machine learning", "deep learning", "nlp", "natural language processing",
    "computer vision", "data science", "data analytics",
    "data engineering", "data warehousing", "data visualization",
    "big data", "business intelligence", "bi development",
    "etl", "data pipeline", "data lake",
    "predictive analytics", "recommendation engine",
    "cybersecurity", "it security", "network security",
    "penetration testing", "vulnerability assessment",
    "security audit", "soc ", "siem",
    "identity management", "access management",
    "managed security", "information security",
    "application security", "cloud security",
    "it services", "it consulting", "it management",
    "it outsourcing", "it support", "managed it",
    "it strategy", "technology consulting",
    "systems integration", "enterprise software",
    "digital transformation", "it staff augmentation",
    "software outsourcing", "offshore development", "nearshore",
    "dedicated team", "staff augmentation",
    "technical support", "helpdesk",
    "qa ", "quality assurance", "software testing",
    "test automation", "manual testing",
    "performance testing", "load testing",
    "security testing", "usability testing",
    "regression testing", "mobile testing",
    "erp", "crm development", "crm consulting",
    "salesforce", "hubspot development", "zoho",
    "sap ", "oracle ", "microsoft dynamics",
    "odoo", "netsuite",
    "enterprise resource planning", "business process management",
    "blockchain", "web3", "smart contract",
    "nft", "defi", "cryptocurrency",
    "iot", "internet of things",
    "embedded", "firmware",
    "ar/vr", "augmented reality", "virtual reality",
    "mixed reality", "metaverse",
    "robotics", "robotic process automation", "rpa",
    "3d modeling", "3d printing",
    "wearable", "edge computing",
    "game development", "game design", "unity",
    "unreal engine", "gamification",
    "ux design", "ui design", "ux/ui", "ui/ux",
    "user experience", "user interface",
    "product design", "interaction design",
    "design system", "figma", "sketch",
    "wireframing", "prototyping",
    "responsive design", "accessibility design",
    "telecommunications", "voip", "unified communications",
    "network engineering", "network administration",
    "wireless", "5g",
]

NON_TECH_ONLY = [
    "accounting firm", "accounting", "bookkeeping",
    "law firm", "legal services", "legal consulting",
    "tax preparation", "tax consulting", "audit",
    "financial advisory", "wealth management",
    "insurance brokerage", "insurance",
    "real estate agency", "real estate", "property management",
    "mortgage", "title services",
    "construction company", "construction", "general contractor",
    "plumbing", "electrical contractor", "hvac",
    "roofing", "flooring", "painting contractor",
    "carpentry", "masonry", "demolition",
    "excavation", "paving", "concrete",
    "landscaping", "lawn care", "tree service",
    "pool construction", "fencing",
    "architecture firm", "architecture",
    "interior design firm", "interior design",
    "industrial design", "landscape architecture",
    "urban planning", "structural engineering",
    "civil engineering", "mechanical engineering",
    "environmental engineering",
    "traditional advertising", "print advertising",
    "billboard", "outdoor advertising",
    "radio advertising", "tv advertising", "television advertising",
    "direct mail", "print media",
    "newspaper", "magazine advertising",
    "media buying", "media planning",
    "public relations", "pr firm", "pr agency",
    "crisis management", "reputation management",
    "corporate communications",
    "branding agency", "branding", "brand strategy",
    "logo design", "graphic design", "print design",
    "packaging design", "signage",
    "trade show", "exhibit design",
    "photography", "videography", "video production",
    "animation studio", "motion graphics",
    "copywriting", "creative agency",
    "advertising agency",
    "seo", "search engine optimization",
    "ppc", "pay per click", "search engine marketing", "sem",
    "conversion rate optimization", "cro",
    "marketing automation", "email marketing",
    "programmatic advertising", "google ads",
    "web analytics", "google analytics",
    "a/b testing", "growth hacking",
    "app store optimization", "aso",
    "martech",
    "social media marketing", "social media management",
    "content marketing", "content strategy",
    "digital marketing", "online marketing",
    "influencer marketing", "affiliate marketing",
    "digital advertising", "display advertising",
    "video marketing", "youtube marketing",
    "digital strategy", "inbound marketing",
    "localization", "translation",
    "event planning", "event management",
    "conference planning", "trade show management",
    "catering", "venue management",
    "wedding planning", "hospitality consulting",
    "staffing agency", "recruiting", "recruitment",
    "executive search", "headhunting",
    "hr consulting", "human resources",
    "payroll", "benefits administration",
    "training", "corporate training",
    "leadership development", "coaching",
    "outplacement",
    "logistics", "supply chain", "freight",
    "warehousing", "distribution", "shipping",
    "fleet management", "courier",
    "import export", "customs brokerage",
    "janitorial", "cleaning service", "facility management",
    "pest control", "waste management",
    "security guard", "physical security",
    "medical practice", "dental practice",
    "veterinary", "pharmacy",
    "home health", "nursing",
    "physical therapy", "occupational therapy",
    "mental health", "counseling",
    "management consulting", "business consulting",
    "strategy consulting", "operations consulting",
    "market research", "business research",
    "customer research", "survey",
    "tutoring", "educational services",
    "print shop", "printing services",
    "manufacturing", "machining",
    "metal fabrication", "woodworking",
    "textile", "garment", "food production",
]

def _norm(text):
    if not text:
        return ""
    if isinstance(text, list):
        parts = []
        for item in text:
            if isinstance(item, dict):
                parts.append(item.get("name", ""))
            else:
                parts.append(str(item))
        text = " ".join(parts)
    return re.sub(r"\s+", " ", str(text).lower()).strip()


def _has_any(text, keywords):
    for kw in keywords:
        if kw in text:
            return True
    return False


def _is_tech_service(name):
    n = name.lower().strip()
    return any(kw in n for kw in TECH_KEYWORDS)


def _calc_tech_split(services):
    if not services or not isinstance(services, list):
        return 0, 0, 0

    tech_sum = 0
    non_tech_sum = 0
    has_pct = False

    for svc in services:
        if not isinstance(svc, dict):
            continue
        name = svc.get("name", "")
        pct_str = svc.get("pct", "")

        pct_val = 0
        if pct_str:
            m = re.search(r"(\d+)", str(pct_str))
            if m:
                pct_val = int(m.group(1))
                has_pct = True

        if _is_tech_service(name):
            tech_sum += pct_val
        else:
            non_tech_sum += pct_val

    if has_pct and (tech_sum + non_tech_sum) > 0:
        total = tech_sum + non_tech_sum
        return round((tech_sum / total) * 100), round((non_tech_sum / total) * 100), len(services)

    tech_count = sum(1 for s in services if isinstance(s, dict) and _is_tech_service(s.get("name", "")))
    total = len([s for s in services if isinstance(s, dict)])
    if total > 0:
        t_pct = round((tech_count / total) * 100)
        return t_pct, 100 - t_pct, total

    return 0, 0, 0


def _bucket_from_split(tech_pct, non_tech_pct):
    """tech > non_tech → green; tied → red; non_tech > tech → yellow."""
    if tech_pct > non_tech_pct:
        return "green"
    if tech_pct == non_tech_pct:
        return "red"
    return "yellow"


def classify_company(company):
    services = company.get("services", [])
    desc = _norm(company.get("description", ""))
    ind_text = _norm(company.get("industry", []))
    focus = _norm(company.get("focus_area", []))
    client = _norm(company.get("client_type", []))
    all_text = f"{_norm(services)} {ind_text} {desc} {focus}"

    if client:
        b2c_only = ("consumer" in client) and not any(
            kw in client for kw in ["business", "enterprise", "midmarket", "small business", "b2b"]
        )
        if b2c_only:
            return {
                "bucket": "yellow",
                "reason": "B2C-only company",
                "is_b2b": False,
                "tech_pct": 0,
                "non_tech_pct": 0,
            }

    tech_pct, non_tech_pct, svc_count = _calc_tech_split(services)

    if svc_count > 0:
        bucket = _bucket_from_split(tech_pct, non_tech_pct)
        reason = f"tech {tech_pct}% / non-tech {non_tech_pct}%"
        return {
            "bucket": bucket,
            "reason": reason,
            "is_b2b": True,
            "tech_pct": tech_pct,
            "non_tech_pct": non_tech_pct,
        }

    has_tech = _has_any(all_text, TECH_KEYWORDS)
    has_hard_non_tech = _has_any(all_text, NON_TECH_ONLY)

    if has_tech and not has_hard_non_tech:
        return {
            "bucket": "red",
            "reason": "Tech in description but no services data (needs review)",
            "is_b2b": True,
            "tech_pct": -1,
            "non_tech_pct": -1,
        }

    if has_hard_non_tech and not has_tech:
        return {
            "bucket": "yellow",
            "reason": "Non-tech company (no services data)",
            "is_b2b": True,
            "tech_pct": 0,
            "non_tech_pct": 0,
        }

    return {
        "bucket": "red",
        "reason": "Insufficient data (needs review)",
        "is_b2b": True,
        "tech_pct": -1,
        "non_tech_pct": -1,
    }


def filter_enriched(companies, keep_buckets=("green",)):
    passed, rejected = [], []
    stats = {"green": 0, "red": 0, "yellow": 0}

    for co in companies:
        result = classify_company(co)
        co["icp_bucket"] = result["bucket"]
        co["icp_reason"] = result["reason"]
        co["is_b2b"] = result["is_b2b"]
        co["tech_pct"] = result["tech_pct"]
        co["non_tech_pct"] = result["non_tech_pct"]
        stats[result["bucket"]] += 1

        if result["bucket"] in keep_buckets:
            passed.append(co)
        else:
            rejected.append(co)

    return passed, rejected, stats


if __name__ == "__main__" or "enriched_companies" in dir():
    passed, rejected, stats = filter_enriched(enriched_companies, keep_buckets=("green",))

    print(f"\n{'=' * 55}")
    print(f"  ICP FILTER RESULTS")
    print(f"  green tech > non-tech | red tied | yellow non-tech > tech")
    print(f"{'=' * 55}")
    print(f"  Total companies         : {len(enriched_companies)}")
    print(f"  Green  (Fit)            : {stats['green']}")
    print(f"  Red    (Mixed / review) : {stats['red']}")
    print(f"  Yellow (Not Fit)        : {stats['yellow']}")
    print(f"  Kept                    : {len(passed)}")
    print(f"{'=' * 55}")

    if rejected:
        print(f"\n  Sample rejected:")
        for c in rejected[:10]:
            t = c.get("tech_pct", "?")
            nt = c.get("non_tech_pct", "?")
            print(f"    {c.get('company_name', '?'):40s}  [{c['icp_bucket']}] tech {t}% / non-tech {nt}% — {c['icp_reason']}")

    if passed:
        print(f"\n  Sample passed:")
        for c in passed[:5]:
            print(f"    {c.get('company_name', '?'):40s}  tech {c.get('tech_pct', '?')}% / non-tech {c.get('non_tech_pct', '?')}%")

    filtered_companies = passed
    print(f"\n  filtered_companies now has {len(filtered_companies)} leads.")


  ICP FILTER RESULTS
  green tech > non-tech | red tied | yellow non-tech > tech
  Total companies         : 50
  Green  (Fit)            : 50
  Red    (Mixed / review) : 0
  Yellow (Not Fit)        : 0
  Kept                    : 50

  Sample passed:
    MOBIKASA                                  tech 100% / non-tech 0%
    Elogic Commerce                           tech 85% / non-tech 15%
    Classy Llama                              tech 90% / non-tech 10%
    Codup                                     tech 100% / non-tech 0%
    Transform Agency                          tech 80% / non-tech 20%

  filtered_companies now has 50 leads.


In [18]:
import re
import csv
import html as html_mod

OUTPUT_FILE = "clutch_enriched_leads.csv"

def clean_text(val):
    if not isinstance(val, str):
        return val
    val = re.sub(r'<[^>]+>', ' ', val)
    val = html_mod.unescape(val)
    val = val.replace('\u2018', "'").replace('\u2019', "'")
    val = val.replace('\u201C', '"').replace('\u201D', '"')
    val = val.replace('\u2013', '-').replace('\u2014', '-')
    val = val.replace('\u2026', '...')
    val = val.replace('\u00A0', ' ')
    return re.sub(r'\s+', ' ', val).strip()

def flatten_items(items):
    if not items:
        return ""
    if isinstance(items[0], dict) and "name" in items[0]:
        parts = []
        for item in items:
            name = item.get("name", "")
            pct = item.get("pct", "")
            parts.append(f"{name} ({pct})" if pct else name)
        return "; ".join(parts)
    return "; ".join(str(i) for i in items)

def flatten_focus(focus_list):
    if not focus_list:
        return ""
    if isinstance(focus_list[0], dict) and "category" in focus_list[0]:
        sections = []
        for group in focus_list:
            cat = group.get("category", "")
            items = flatten_items(group.get("items", []))
            sections.append(f"[{cat}] {items}")
        return " | ".join(sections)
    return flatten_items(focus_list)

COLUMNS = [
    "company_name", "company_url", "company_website", "source_category",
    "clutch_rating", "project_min_cost", "hourly_rate", "company_size",
    "year_founded", "description", "services", "industry",
    "client_type", "focus_area", "locations", "languages_services",
    "timezones_services",
]

rows = []
for company in filtered_companies:
    row = []
    for col in COLUMNS:
        val = company.get(col, "")
        if col == "focus_area":
            cell = clean_text(flatten_focus(val))
        elif col in ("services", "industry", "client_type"):
            cell = clean_text(flatten_items(val))
        elif col in ("locations", "languages_services", "timezones_services"):
            cell = clean_text("; ".join(val) if isinstance(val, list) else str(val))
        else:
            cell = clean_text(val if val else "")
        row.append(cell if cell else "Not found")
    rows.append(row)

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(COLUMNS)
    writer.writerows(rows)

print(f"Saved {len(rows)} green leads to {OUTPUT_FILE}")

Saved 50 green leads to clutch_enriched_leads.csv


In [19]:
pip install anthropic google-auth google-cloud-secret-manager

Note: you may need to restart the kernel to use updated packages.


In [20]:
import google.auth
from google.auth import impersonated_credentials
from google.cloud import secretmanager

SA_EMAIL = "export-backend-sa@tooltoenquire.iam.gserviceaccount.com"
SECRET_NAME = "projects/tooltoenquire/secrets/Anthropic_API_Key/versions/latest"

base_creds, _ = google.auth.default()
sa_creds = impersonated_credentials.Credentials(
    source_credentials=base_creds,
    target_principal=SA_EMAIL,
    target_scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
sm_client = secretmanager.SecretManagerServiceClient(credentials=sa_creds)
api_key = sm_client.access_secret_version(
    request={"name": SECRET_NAME}
).payload.data.decode("UTF-8")
print(f"✓ API key loaded (length: {len(api_key)})")

✓ API key loaded (length: 108)


In [21]:
SYSTEM_PROMPT = """You are a B2B sales analyst qualifying leads for a technology services provider. Analyze each Clutch.co company profile and decide if it's Fit, Mixed (requires review), or Not Fit. Return only valid JSON. No prose, no markdown, no preamble.

# 1. Definitions

## FIT (Green) — tech_pct > non_tech_pct
A company whose **primary revenue source** is technical services or products. The company sells engineering, infrastructure, or data work to other businesses.

A Fit company:
- Builds or ships **software**: custom applications, SaaS platforms, web/mobile apps, APIs, embedded systems, firmware
- Delivers **cloud or infrastructure** work: AWS/Azure/GCP consulting, DevOps, cloud migration, platform engineering, Kubernetes
- Provides **data, AI, or ML** services: data engineering, analytics, ML models, computer vision, NLP, BI dashboards
- Operates in **cybersecurity**: pen testing, security audits, managed security, compliance engineering
- Offers **IT services** with technical depth: IT consulting, managed IT, systems integration, ERP/CRM implementation
- Provides **technical staff augmentation**: dedicated engineering teams, offshore development, software outsourcing
- Sells **specific technologies as service**: blockchain, IoT, AR/VR, game development, robotics

The buyer is technical (CTO, VP Engineering, IT director) or digital-first. Revenue is billed for technical work — code shipped, infrastructure built, data delivered.

## MIXED (Yellow — review required) — tech_pct == non_tech_pct
A company tied between tech and non-tech revenue (50/50, 40/40 with 20 unknown, etc.). Ambiguous from `services` and `focus_area` alone — flag for human/AI second pass with full description and context.

## NOT FIT (Red) — tech_pct < non_tech_pct
A company whose **primary revenue source is non-technical**, regardless of whether they have a website, use computers, or list one tech-sounding service.

A Not Fit company:
- **Professional services**: accounting, legal, tax, audit, financial advisory, insurance, real estate, mortgage
- **Construction & trades**: general contracting, plumbing, HVAC, roofing, flooring, landscaping, paving, concrete
- **Architecture & traditional engineering**: architecture firms, interior design, civil/mechanical/structural engineering (NOT software engineering)
- **Marketing & creative agencies**: traditional advertising, branding, logo/print/graphic design, PR firms, copywriting, photography, video production, animation studios, **SEO, PPC, SEM, content/email/social/digital marketing, performance marketing, demand generation, CRO, ASO, MarTech, media planning & buying, growth hacking, marketing analytics**
- **Event & hospitality**: event planning, catering, wedding planning, venue management
- **Staffing & HR**: recruiting, executive search, HR consulting, payroll, training, coaching, sales outsourcing, lead gen, telemarketing, call center / BPO
- **Logistics & operations**: supply chain, warehousing, freight, shipping, fleet management
- **Facility & physical services**: janitorial, cleaning, pest control, security guards, waste management
- **Healthcare practices**: medical, dental, veterinary, pharmacy, therapy, counseling
- **Generic consulting**: management consulting, strategy consulting, business consulting, market research
- **Manufacturing & physical goods**: machining, metal fabrication, textile, food production
- **Education**: tutoring, traditional educational services

## Critical Rule
**A single tech-sounding side service does not flip a Not Fit into a Fit.** A construction firm with "Web Development - 10%" stays Not Fit. A marketing agency with "CRM Consulting - 10%" stays Not Fit.

## Governing Rule (the only classification rule)
- tech_pct  >  non_tech_pct  → Fit
- tech_pct  == non_tech_pct  → Mixed
- tech_pct  <  non_tech_pct  → Not Fit

Score is computed for ranking inside each bucket but does NOT change the bucket assignment.

# 2. Service Buckets (semantic judgment, not just substring match)

HARD-TECH (counts toward tech_pct AND hard_tech_pct):
software/app/web/mobile development, full stack, backend, frontend, MVP, API, microservices, SaaS/PaaS, cloud (AWS/Azure/GCP/Kubernetes/Docker), DevOps, AI/ML/data science/analytics/engineering, BI, ETL, cybersecurity, IT services/consulting/support/outsourcing, QA/testing, ERP/CRM (Salesforce/SAP/Oracle), blockchain, IoT, AR/VR, embedded, game dev, systems integration, digital transformation, staff augmentation, and any specific programming language/framework (React, Python, Java, Flutter, etc.). Vertical wrappers like "Healthcare Software" or "Software for Law Firms" are still HARD-TECH — bucket by the primary noun (software), not the vertical.

SOFT-TECH (counts toward tech_pct but NOT hard_tech_pct):
UX/UI design, product design, interaction design, wireframing, prototyping, design systems, accessibility design, responsive design. **Design only — marketing is not soft-tech.**

NON-TECH (counts toward non_tech_pct):
- Professional services: accounting, legal, tax, audit, financial advisory, insurance, real estate, mortgage
- Construction & trades: plumbing, HVAC, roofing, flooring, landscaping, paving, concrete, all contracting
- Traditional engineering: architecture, interior design, civil/mechanical/structural engineering
- Creative agencies: traditional advertising, print/PR, branding, logo/graphic/print/packaging design, photography, videography, video production, animation, copywriting, creative agency, advertising agency
- **Marketing services**: SEO, PPC, SEM, CRO, ASO, MarTech, marketing automation, email marketing, content marketing, social media marketing, digital marketing, performance marketing, demand generation, growth hacking, programmatic advertising, web/marketing analytics, influencer marketing, affiliate marketing, media planning & buying, localization, translation
- Events & hospitality: event planning/management, catering, wedding planning
- Staffing: recruiting, HR, payroll, training, coaching, sales outsourcing, lead generation, appointment setting, cold calling, telemarketing, call center services, BPO
- Logistics: supply chain, warehousing, freight, shipping, fleet management
- Facility services: janitorial, cleaning, pest control, security guards, waste management
- Healthcare practices: medical, dental, veterinary, pharmacy
- Generic consulting: management/business/strategy consulting, market research
- Education: tutoring, educational services
- Manufacturing: machining, metal fabrication, textile, food production

UNKNOWN: services matching neither list. Recorded but never moved into a bucket.

# 3. Pipeline (deterministic, in order)

Step 1 — Extract services from `services` and `focus_area` fields only. Use focus_area percentages as weights when present; otherwise weight services equally. Do not infer services from description, industry, or company name. EXCEPTION: if `services` is empty AND description_quality == "rich", you may extract services from description but mark them inferred and floor confidence at medium.

Step 2 — Classify description quality:
- rich: ≥ 2 sentences AND mentions specific tech/methodology/vertical/deliverable/client.
- thin: exists but generic marketing language only ("innovative solutions", "trusted partner", "cutting-edge").
- empty: null, missing, or < 10 meaningful words.

Step 3 — Bucket each service semantically. Compute tech_pct, non_tech_pct, unknown_pct (sum to 100.0). Track hard_tech_pct separately (HARD-TECH only, no soft-tech).

Step 4 — Classify (THE ONLY RULE):
- tech_pct >  non_tech_pct  → "Fit"
- tech_pct == non_tech_pct  → "Mixed"
- tech_pct <  non_tech_pct  → "Not Fit"

Step 5 — Score (for ranking inside the bucket, NOT for classification):
penalty = 0.5 × max(0, non_tech_pct − tech_pct)
score = clamp(round(tech_pct − penalty), 1, 100)

Worked examples:
- 70/30 → 70 − 0 = 70 (Fit)
- 60/40 → 60 − 0 = 60 (Fit)
- 50/50 → 50 − 0 = 50 (Mixed — tied)
- 40/60 → 40 − 10 = 30 (Not Fit)
- 0/100 → 0 − 50 → 1 (Not Fit)

Step 6 — Confidence (start high, apply each rule):
- unknown_pct ≥ 20: downgrade 1
- unknown_pct ≥ 40: downgrade 1 more
- description_quality == "thin": floor at medium
- services inferred from description: floor at medium
- services AND description both empty: force low
- abs(tech_pct − non_tech_pct) < 10 (and not exactly tied): downgrade 1
Steps: high → medium → low. Never below low.

Step 7 — review_reason and rejection_reason. **Both fields are ALWAYS populated** (no nulls, no empty strings):

For Fit:
- review_reason = "n/a"
- rejection_reason = "n/a"

For Mixed (BOTH fields filled with real values — describes why it's tied AND why it's risky):
- review_reason: pick the most decisive of:
    - "balanced_split" (tech_pct == non_tech_pct, both > 0)
    - "soft_tech_only_review" (hard_tech_pct == 0 AND tech is all soft-tech design)
    - "unknown_dominant_review" (unknown_pct > 50)
    - "thin_description_review" (services empty, description thin/empty)
    - "low_signal_review" (none of the above clearly applies)
- rejection_reason: pick the most decisive of:
    - "balanced_revenue_split" (tied 50/50, neither side dominates)
    - "no_hard_tech_signal" (hard_tech_pct == 0)
    - "thin_description_no_services" (services + description both weak)
    - "ambiguous_classification" (default for tied without clearer driver)

For Not Fit:
- review_reason = "n/a"
- rejection_reason: pick the most decisive of:
    - "non_tech_dominant" (non_tech_pct > tech_pct, the normal case)
    - "thin_description_no_services" (services empty + description thin/empty)
    - "all_marketing" (100% marketing/SEO/PPC, zero engineering)

# 4. Output Schema (STRICT — no extra keys, no markdown, no code fence, NO null values)

{
  "company_name": "string",
  "classification": "Fit" | "Mixed" | "Not Fit",
  "score": <integer 1-100>,
  "confidence": "high" | "medium" | "low",
  "review_reason": "<one of the values above, NEVER null, use 'n/a' if not applicable>",
  "rejection_reason": "<one of the values above, NEVER null, use 'n/a' if not applicable>",
  "description_quality": "rich" | "thin" | "empty",
  "reasoning": "<2-3 sentences: composition, classification driver, decisive factor>",
  "tech_services_matched": [<strings>, or ["none"] if no matches],
  "non_tech_services_matched": [<strings>, or ["none"] if no matches],
  "unmatched_services": [<strings>, or ["none"] if no matches],
  "composition": {"tech_pct": <float>, "non_tech_pct": <float>, "unknown_pct": <float>, "hard_tech_pct": <float>}
}

Rules:
- composition sums to 100.0 ±0.1
- hard_tech_pct ≤ tech_pct
- review_reason and rejection_reason are NEVER null and NEVER empty strings — always a string value. For Mixed, BOTH are populated with real reasons (not "n/a"). For Fit, both are "n/a". For Not Fit, review_reason is "n/a" and rejection_reason is a real reason.
- All array fields use ["none"] when empty, never []
- Classification follows the comparison rule strictly: tech_pct > non_tech_pct → Fit, tied → Mixed, tech_pct < non_tech_pct → Not Fit
- Every matched service must trace to a literal input
- No banned phrases ("Overall", "It's worth noting", "Interestingly", hedges)
- No restating the company name
- No `score_breakdown`, no `caps_applied`, no `base_score` — those fields are removed

# 5. Examples

## Pure tech (Fit)
Input: {"company_name":"TechForge","services":"Custom Software | Mobile App | Cloud Consulting","focus_area":"Custom Software - 50% | Mobile App - 30% | Cloud - 20%","description":"We build scalable web and mobile apps with deep AWS expertise."}
Output: {"company_name":"TechForge","classification":"Fit","score":100,"confidence":"high","review_reason":"n/a","rejection_reason":"n/a","description_quality":"rich","reasoning":"100% hard-tech composition (custom software, mobile app, cloud). Tech leads non-tech 100 to 0 — Fit. Score 100.","tech_services_matched":["custom software","mobile app","cloud consulting"],"non_tech_services_matched":["none"],"unmatched_services":["none"],"composition":{"tech_pct":100.0,"non_tech_pct":0.0,"unknown_pct":0.0,"hard_tech_pct":100.0}}

## Tech-leaning 70/30 (Fit)
Input: {"company_name":"BuildRight","services":"Custom Software | Cloud Migration | UX Design | Branding","focus_area":"Software - 40% | Cloud - 20% | UX - 10% | Branding - 30%","description":"Engineering-led product studio for SaaS founders."}
Output: {"company_name":"BuildRight","classification":"Fit","score":70,"confidence":"high","review_reason":"n/a","rejection_reason":"n/a","description_quality":"rich","reasoning":"70% tech (software, cloud, UX) vs 30% non-tech (branding). Tech leads — Fit. Score 70.","tech_services_matched":["custom software","cloud migration","ux design"],"non_tech_services_matched":["branding"],"unmatched_services":["none"],"composition":{"tech_pct":70.0,"non_tech_pct":30.0,"unknown_pct":0.0,"hard_tech_pct":60.0}}

## Tied 50/50 (Mixed — both reasons filled)
Input: {"company_name":"BlendCo","services":"Web Development | Branding | Cloud Consulting | Print Design","focus_area":"Web - 30% | Branding - 25% | Cloud - 20% | Print - 25%","description":"Full-service studio combining engineering and brand work."}
Output: {"company_name":"BlendCo","classification":"Mixed","score":50,"confidence":"medium","review_reason":"balanced_split","rejection_reason":"balanced_revenue_split","description_quality":"rich","reasoning":"50% tech (web, cloud) vs 50% non-tech (branding, print). Tied — Mixed. Engineering depth needs review to determine primary revenue driver.","tech_services_matched":["web development","cloud consulting"],"non_tech_services_matched":["branding","print design"],"unmatched_services":["none"],"composition":{"tech_pct":50.0,"non_tech_pct":50.0,"unknown_pct":0.0,"hard_tech_pct":50.0}}

## Soft-tech tied with non-tech (Mixed)
Input: {"company_name":"Pixel Co","services":"UX Design | UI Design | Branding | Logo Design","focus_area":"UX - 25% | UI - 25% | Branding - 25% | Logo - 25%","description":"Design-led studio working across digital and print."}
Output: {"company_name":"Pixel Co","classification":"Mixed","score":50,"confidence":"medium","review_reason":"soft_tech_only_review","rejection_reason":"no_hard_tech_signal","description_quality":"rich","reasoning":"50% soft-tech design (UX/UI) vs 50% non-tech (branding, logo). Tied — Mixed. Zero hard-tech signal; engineering capability unknown.","tech_services_matched":["ux design","ui design"],"non_tech_services_matched":["branding","logo design"],"unmatched_services":["none"],"composition":{"tech_pct":50.0,"non_tech_pct":50.0,"unknown_pct":0.0,"hard_tech_pct":0.0}}

## Creative agency 30/70 (Not Fit)
Input: {"company_name":"Apex Digital","services":"Graphic Design | Web Development | Print Design | Logo Design","focus_area":"Graphic - 40% | Web - 30% | Print - 20% | Logo - 10%","description":"Creative agency offering design and digital solutions."}
Output: {"company_name":"Apex Digital","classification":"Not Fit","score":10,"confidence":"medium","review_reason":"n/a","rejection_reason":"non_tech_dominant","description_quality":"thin","reasoning":"30% tech (web) vs 70% non-tech (graphic, print, logo). Non-tech leads — Not Fit. Penalty: 30 − 0.5 × 40 = 10.","tech_services_matched":["web development"],"non_tech_services_matched":["graphic design","print design","logo design"],"unmatched_services":["none"],"composition":{"tech_pct":30.0,"non_tech_pct":70.0,"unknown_pct":0.0,"hard_tech_pct":30.0}}

## Marketing agency (Not Fit)
Input: {"company_name":"Growthly","services":"Digital Marketing | SEO | PPC | Content Marketing","focus_area":"Marketing - 35% | SEO - 30% | PPC - 20% | Content - 15%","description":"Performance marketing for DTC brands."}
Output: {"company_name":"Growthly","classification":"Not Fit","score":1,"confidence":"high","review_reason":"n/a","rejection_reason":"all_marketing","description_quality":"rich","reasoning":"100% non-tech composition (digital marketing, SEO, PPC, content). Non-tech leads 100 to 0 — Not Fit. Score clamped to 1.","tech_services_matched":["none"],"non_tech_services_matched":["digital marketing","seo","ppc","content marketing"],"unmatched_services":["none"],"composition":{"tech_pct":0.0,"non_tech_pct":100.0,"unknown_pct":0.0,"hard_tech_pct":0.0}}

## Mixed marketing + tech 35/65 (Not Fit)
Input: {"company_name":"SemNexus","services":"Mobile App Development | SEO | ASO | Social Media Ads | App Analytics","focus_area":"Mobile App - 35% | SEO - 25% | ASO - 15% | Social Ads - 15% | Analytics - 10%","description":"App marketing agency with light development capability."}
Output: {"company_name":"SemNexus","classification":"Not Fit","score":20,"confidence":"high","review_reason":"n/a","rejection_reason":"non_tech_dominant","description_quality":"rich","reasoning":"35% hard-tech (mobile app development) vs 65% non-tech (SEO, ASO, social ads, marketing analytics). Non-tech leads — Not Fit. Single tech service does not flip classification.","tech_services_matched":["mobile app development"],"non_tech_services_matched":["seo","aso","social media ads","app analytics"],"unmatched_services":["none"],"composition":{"tech_pct":35.0,"non_tech_pct":65.0,"unknown_pct":0.0,"hard_tech_pct":35.0}}

## Vertical wrapper (Fit)
Input: {"company_name":"MedStack","services":"Healthcare Software Solutions | HIPAA-Compliant Cloud Architecture | Medical Data Analytics","focus_area":"Healthcare Software - 50% | Cloud - 30% | Data - 20%","description":"We engineer secure software platforms for hospital systems."}
Output: {"company_name":"MedStack","classification":"Fit","score":100,"confidence":"high","review_reason":"n/a","rejection_reason":"n/a","description_quality":"rich","reasoning":"All three services bucket as hard-tech by primary noun (software, cloud, data analytics); healthcare is the vertical. Tech leads 100 to 0 — Fit.","tech_services_matched":["software","cloud","data analytics"],"non_tech_services_matched":["none"],"unmatched_services":["none"],"composition":{"tech_pct":100.0,"non_tech_pct":0.0,"unknown_pct":0.0,"hard_tech_pct":100.0}}
"""

print(f"Prompt loaded: {len(SYSTEM_PROMPT)} chars")

Prompt loaded: 18071 chars


In [22]:
import json
import time
import anthropic

CLAUDE_MODEL = "claude-haiku-4-5-20251001"
claude = anthropic.Anthropic(api_key=api_key)

# Only fields the prompt actually reads. Extra fields (hourly_rate, company_size,
# year_founded, locations, etc.) inflate per-call token cost and risk the model
# treating them as classification signals despite the prompt's explicit ban on
# inferring services from non-service fields.
INPUT_FIELDS = (
    "company_name", "description", "services",
    "industry", "client_type", "focus_area",
)


def classify_with_ai(company, max_retries=5):
    payload = {k: company.get(k) for k in INPUT_FIELDS}
    user_msg = json.dumps(payload, ensure_ascii=False, default=str)

    for attempt in range(max_retries + 1):
        try:
            response = claude.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1000,
                system=[
                    {
                        "type": "text",
                        "text": SYSTEM_PROMPT,
                        "cache_control": {"type": "ephemeral"},
                    }
                ],
                messages=[{"role": "user", "content": user_msg}],
            )
            text = response.content[0].text.strip()
            if text.startswith("```"):
                text = text.lstrip("`")
                if text.lower().startswith("json"):
                    text = text[4:]
                text = text.strip().rstrip("`").strip()
            return json.loads(text)

        except anthropic.RateLimitError:
            wait = 30 + (attempt * 15)
            if attempt < max_retries:
                time.sleep(wait)
                continue
            return {"company_name": company.get("company_name"), "classification": "ERROR", "score": None, "error": "rate_limit_exhausted"}

        except (anthropic.APIError, anthropic.APIConnectionError) as e:
            wait = 5 * (2 ** attempt)
            if attempt < max_retries:
                time.sleep(wait)
                continue
            return {"company_name": company.get("company_name"), "classification": "ERROR", "score": None, "error": f"api_error: {e}"}

        except json.JSONDecodeError as e:
            if attempt < max_retries:
                time.sleep(2)
                continue
            return {"company_name": company.get("company_name"), "classification": "ERROR", "score": None, "error": f"json_parse: {e}"}

    return {"company_name": company.get("company_name"), "classification": "ERROR", "score": None, "error": "unknown"}


print("✓ classify_with_ai defined (rate-limit aware, max_retries=5, prompt caching enabled)")

✓ classify_with_ai defined (rate-limit aware, max_retries=5, prompt caching enabled)


In [23]:
import time
import json
import pandas as pd
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# === EDIT THESE PER RUN ===
CSV_PATH = "clutch_enriched_leads.csv"
MY_LABEL = "tech"          # "tech", "non_tech", or "mixed"
MAX_WORKERS = 10
RESULTS_PATH = f"results_{MY_LABEL}.json"
# ==========================

SOURCE_COLUMNS = [
    "company_name", "company_url", "company_website", "source_category",
    "clutch_rating", "project_min_cost", "hourly_rate", "company_size",
    "year_founded", "description", "services", "industry", "client_type",
    "focus_area", "locations", "languages_services", "timezones_services",
]


def process_row(row, label, max_retries=3):
    last_err = None
    for attempt in range(max_retries):
        try:
            ai = classify_with_ai(row) or {}
            comp = ai.get("composition") or {}

            out = {col: row.get(col, "") for col in SOURCE_COLUMNS}
            out.update({
                "my_label": label,
                "classification": ai.get("classification"),
                "score": ai.get("score"),
                "confidence": ai.get("confidence"),
                "rejection_reason": ai.get("rejection_reason"),
                "description_quality": ai.get("description_quality"),
                "reasoning": ai.get("reasoning"),
                "tech_services_matched": ai.get("tech_services_matched") or [],
                "non_tech_services_matched": ai.get("non_tech_services_matched") or [],
                "unmatched_services": ai.get("unmatched_services") or [],
                "tech_pct": comp.get("tech_pct"),
                "non_tech_pct": comp.get("non_tech_pct"),
                "unknown_pct": comp.get("unknown_pct"),
                "hard_tech_pct": comp.get("hard_tech_pct"),
            })
            return out
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    raise last_err


def classify_companies(companies, label, max_workers=10):
    results = [None] * len(companies)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_row, row, label): i for i, row in enumerate(companies)}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"Classifying {label}"):
            i = futures[future]
            try:
                results[i] = future.result()
            except Exception as e:
                err_row = {col: companies[i].get(col, "") for col in SOURCE_COLUMNS}
                err_row.update({
                    "my_label": label,
                    "classification": "ERROR",
                    "reasoning": f"Exception after retries: {e}",
                })
                results[i] = err_row

    return results


def print_summary(results, label, source_path, elapsed):
    fit     = sum(1 for r in results if r["classification"] == "Fit")
    mixed   = sum(1 for r in results if r["classification"] == "Mixed")
    not_fit = sum(1 for r in results if r["classification"] == "Not Fit")
    errors  = sum(1 for r in results if r["classification"] == "ERROR")
    expected = {"tech": "Fit", "non_tech": "Not Fit", "mixed": "Mixed"}.get(label)

    print(f"\n{'=' * 60}")
    print(f"  RESULTS — {source_path} (label: {label})")
    print(f"{'=' * 60}")
    print(f"  Total    : {len(results)}")
    print(f"  Fit      : {fit}")
    print(f"  Mixed    : {mixed}")
    print(f"  Not Fit  : {not_fit}")
    print(f"  Errors   : {errors}")
    print(f"  Elapsed  : {elapsed:.1f}s ({elapsed/len(results):.2f}s per company)")

    if expected:
        correct = sum(1 for r in results if r["classification"] == expected)
        scored_total = len(results) - errors
        if scored_total:
            print(f"  Accuracy : {correct}/{scored_total} = {correct/scored_total:.1%} (expected {expected})")

        disagreements = [r for r in results if r["classification"] not in (expected, "ERROR")]
        if disagreements:
            print(f"\n  DISAGREEMENTS ({len(disagreements)}):")
            for r in disagreements:
                print(f"\n    {r['company_name']}  ({r.get('industry') or '—'})")
                print(f"      Verdict       : {r['classification']}  score={r['score']}  conf={r['confidence']}")
                print(f"      Description   : {r['description_quality']}")
                print(f"      Rejection     : {r['rejection_reason']}")
                print(f"      Composition   : tech={r['tech_pct']}%  non-tech={r['non_tech_pct']}%  unknown={r['unknown_pct']}%  hard-tech={r['hard_tech_pct']}%")
                print(f"      Tech services : {r['tech_services_matched'] or '—'}")
                print(f"      Non-tech svcs : {r['non_tech_services_matched'] or '—'}")
                print(f"      Unmatched     : {r['unmatched_services'] or '—'}")
                print(f"      Reasoning     : {r['reasoning']}")


# ---------- run ----------

df = pd.read_csv(CSV_PATH)
companies = df.fillna("").astype(str).to_dict(orient="records")
print(f"Loaded {len(companies)} companies from {CSV_PATH} (label: {MY_LABEL})\n")

start = time.time()
results = classify_companies(companies, MY_LABEL, MAX_WORKERS)
elapsed = time.time() - start

print_summary(results, MY_LABEL, CSV_PATH, elapsed)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\n✓ Raw results saved → {RESULTS_PATH}")

Loaded 50 companies from clutch_enriched_leads.csv (label: tech)



Classifying tech:   0%|          | 0/50 [00:00<?, ?it/s]


  RESULTS — clutch_enriched_leads.csv (label: tech)
  Total    : 50
  Fit      : 44
  Mixed    : 2
  Not Fit  : 4
  Errors   : 0
  Elapsed  : 53.2s (1.06s per company)
  Accuracy : 44/50 = 88.0% (expected Fit)

  DISAGREEMENTS (6):

    FJ Solutions  (eCommerce (80%); Consumer products & services (5%); Medical (5%); Non-profit (5%); Other industries (5%))
      Verdict       : Mixed  score=50  conf=medium
      Description   : rich
      Rejection     : balanced_revenue_split
      Composition   : tech=50.0%  non-tech=50.0%  unknown=0.0%  hard-tech=50.0%
      Tech services : ['e-commerce development', 'web design', 'shopify development']
      Non-tech svcs : ['ecommerce marketing', 'email marketing', 'search engine optimization', 'cro']
      Unmatched     : ['none']
      Reasoning     : 50% hard-tech (Shopify development, e-commerce development, web design) vs 50% non-tech (ecommerce marketing, SEO, email marketing, CRO). Tied revenue split — Mixed. Primary driver unclear: enginee

In [15]:
import csv as csv_mod
import json

# === EDIT THESE PER RUN ===
MY_LABEL = "tech"
RESULTS_PATH = f"results_{MY_LABEL}.json"
OUTPUT_PATH = f"Final_result_{MY_LABEL}.csv"
# ==========================

SOURCE_COLUMNS = [
    "company_name", "company_url", "company_website", "source_category",
    "clutch_rating", "project_min_cost", "hourly_rate", "company_size",
    "year_founded", "description", "services", "industry", "client_type",
    "focus_area", "locations", "languages_services", "timezones_services",
]

AI_COLUMNS = [
    "my_label", "classification", "score", "confidence",
    "rejection_reason", "description_quality", "reasoning",
    "tech_services_matched", "non_tech_services_matched", "unmatched_services",
    "tech_pct", "non_tech_pct", "unknown_pct", "hard_tech_pct",
]

OUTPUT_COLUMNS = SOURCE_COLUMNS + AI_COLUMNS

LIST_FIELDS = {"tech_services_matched", "non_tech_services_matched", "unmatched_services"}


def flatten_row(row):
    """Convert list fields to '; '-joined strings and fill missing columns."""
    out = {}
    for col in OUTPUT_COLUMNS:
        val = row.get(col, "")
        if col in LIST_FIELDS and isinstance(val, list):
            val = "; ".join(val)
        out[col] = val if val is not None else ""
    return out


def export_to_csv(results, output_path):
    if not results:
        print("No results to save.")
        return
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv_mod.DictWriter(f, fieldnames=OUTPUT_COLUMNS)
        writer.writeheader()
        writer.writerows(flatten_row(r) for r in results)
    print(f"✓ Saved → {output_path}")


# ---------- run ----------

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"Loaded {len(results)} results from {RESULTS_PATH}")
export_to_csv(results, OUTPUT_PATH)

Loaded 49 results from results_tech.json
✓ Saved → Final_result_tech.csv
